# 03 — Modélisation XGBoost

Itérations sur les hyperparamètres et inspection des résidus.

In [ ]:
import sys
from pathlib import Path

sys.path.insert(0, str(Path.cwd().parent))

import joblib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import mean_squared_error, r2_score

from src.config import FEATURES_PARQUET, MODEL_PATH, FEATURE_COLUMNS, TARGET
from src.train import cross_validate, prepare_xy

sns.set_theme(style="whitegrid")

In [ ]:
df = pd.read_parquet(FEATURES_PARQUET)
X, y = prepare_xy(df)
print(X.shape, y.shape)

## Validation croisée temporelle

In [ ]:
folds = cross_validate(X, y, n_splits=5)
pd.DataFrame(folds)

## Importance des variables

In [ ]:
model = joblib.load(MODEL_PATH)
importances = pd.Series(model.feature_importances_, index=FEATURE_COLUMNS)
importances = importances.sort_values(ascending=True)

plt.figure(figsize=(8, 5))
importances.plot.barh()
plt.title("Importance XGBoost (gain)")
plt.show()

## Analyse des résidus

In [ ]:
preds_log = model.predict(X)
preds = np.expm1(preds_log)
true = np.expm1(y)
residuals = true - preds

fig, ax = plt.subplots(1, 2, figsize=(14, 4))
ax[0].scatter(preds, residuals, alpha=0.05, s=4)
ax[0].axhline(0, color='red')
ax[0].set_xlabel("Prédiction (€/m²)")
ax[0].set_ylabel("Résidu")
ax[0].set_title("Résidus vs prédictions")

sns.histplot(residuals, bins=80, ax=ax[1])
ax[1].set_title("Distribution des résidus")
plt.tight_layout()
plt.show()